##  Météo ML DuvTech — Prédiction de la pluie du lendemain

## Description du projet

Ce projet met en œuvre un **algorithme de classification supervisée** capable de prédire **s'il pleuvra demain** à partir de trois relevés météo simples de la journée en cours. Il couvre toute la chaîne d'un projet de Machine Learning, de la création des données jusqu'à une application de bureau utilisable par une personne non technicienne.

| Élément | Détail |
|---|---|
| **Type de problème** | Classification binaire (0 = Beau temps, 1 = Pluie) |
| **Algorithme** | Random Forest (forêt aléatoire, 100 arbres) |
| **Données** | 1 000 jours simulés, cohérents avec le climat d'Haïti |
| **Variables d'entrée** | Température max (°C), Température min (°C), Pluie du jour (mm) |
| **Variable cible** | `Pleuvra_Demain` |
| **Fonctionnement** | 100 % hors-ligne (aucune connexion Internet requise) |
| **Livrables** | Modèle entraîné (`modele_meteo.pkl`) + interface graphique Tkinter |

### Objectifs
1. Générer un jeu de données météo réaliste sans dépendre d'une source en ligne.
2. Entraîner un modèle capable d'apprendre la relation entre la météo du jour et celle du lendemain.
3. Évaluer objectivement ses performances sur des données qu'il n'a jamais vues.
4. Sauvegarder le modèle pour le réutiliser sans ré-entraînement.
5. Le rendre accessible via une interface graphique simple et moderne.

### Démarche (pipeline)
```
Génération des données → Séparation Train/Test → Entraînement → Évaluation → Prédiction → Sauvegarde → Interface graphique
```

### Technologies utilisées
- **Python 3** (distribution Anaconda)
- **NumPy** et **Pandas** : génération et manipulation des données
- **Scikit-learn** : algorithme Random Forest, découpage et métriques
- **Pickle** : sauvegarde du modèle
- **Tkinter** : interface graphique de bureau

## 1. Importation des bibliothèques

On charge les outils nécessaires :
- `numpy` pour les calculs et le tirage de nombres aléatoires ;
- `pandas` pour organiser les données sous forme de tableau (`DataFrame`) ;
- `RandomForestClassifier` : l'algorithme de classification ;
- `train_test_split` : pour séparer les données d'entraînement et de test ;
- `accuracy_score` et `classification_report` : pour mesurer la qualité du modèle.

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

## 2. Création des données (100 % hors-ligne)

Faute de base de données météo disponible localement, on **simule 1 000 jours** de relevés cohérents avec le climat tropical d'Haïti :

| Variable | Méthode de génération | Signification |
|---|---|---|
| `Temp_Max` | Uniforme entre 28 et 35 °C | Température la plus haute de la journée |
| `Temp_Min` | `Temp_Max` moins 5 à 8 °C | Température la plus basse (écart jour/nuit réaliste) |
| `Pluie_Aujourdhui` | Loi exponentielle (moyenne ≈ 2 mm), valeurs < 1 mm ramenées à 0 | Beaucoup de jours secs, quelques jours très pluvieux |

`np.random.seed(42)` fixe le hasard : les mêmes données sont produites à chaque exécution, ce qui rend les résultats **reproductibles**.

In [17]:
print("Génération des données météo locales...")
np.random.seed(42)
nb_jours = 1000

# Génération de températures et de pluies cohérentes pour Haïti
temp_max = np.random.uniform(28, 35, nb_jours)
temp_min = temp_max - np.random.uniform(5, 8, nb_jours)
pluie_aujourdhui = np.random.exponential(scale=2, size=nb_jours)
pluie_aujourdhui[pluie_aujourdhui < 1] = 0  # Beaucoup de jours sans pluie

# Création du DataFrame
donnees = pd.DataFrame(
    {
        "Temp_Max": temp_max,
        "Temp_Min": temp_min,
        "Pluie_Aujourdhui": pluie_aujourdhui,
    }
)

Génération des données météo locales...


### Construction de la variable cible `Pleuvra_Demain`

On applique une **règle logique** inspirée de la météorologie tropicale : *une journée très chaude et déjà humide favorise la pluie le lendemain.*

- Si `Temp_Max > 32 °C` **et** qu'il a plu aujourd'hui → **80 %** de chances de pluie demain.
- Sinon → seulement **20 %** de chances de pluie demain.

Le tirage aléatoire (`np.random.choice`) ajoute volontairement du **bruit** : comme dans la réalité, les mêmes conditions ne donnent pas toujours le même résultat. C'est cette règle cachée que le modèle devra redécouvrir seul.

In [18]:
# S'il fait très chaud et humide aujourd'hui, il risque de pleuvoir demain
probabilite_pluie = (donnees["Temp_Max"] > 32) & (donnees["Pluie_Aujourdhui"] > 0)
donnees["Pleuvra_Demain"] = np.where(
    probabilite_pluie,
    np.random.choice([1, 0], size=nb_jours, p=[0.8, 0.2]),
    np.random.choice([1, 0], size=nb_jours, p=[0.2, 0.8]),
)

print(f"Jeu de données créé : {nb_jours} jours enregistrés.\n")
donnees.head()

Jeu de données créé : 1000 jours enregistrés.



,Temp_Max,Temp_Min,Pluie_Aujourdhui,Pleuvra_Demain
0,30.621781,25.066382,0.000000,0
1,34.655000,28.029297,0.000000,0
2,33.123958,25.505120,4.734345,1
3,32.190609,24.993935,0.000000,1
4,29.092130,21.672447,0.000000,1


## 3. Séparation des données (Train / Test)

- **X** contient les *indices* qui aident à prédire (les caractéristiques) : `Temp_Max`, `Temp_Min`, `Pluie_Aujourdhui`.
- **y** contient la *réponse* à deviner : `Pleuvra_Demain`.

On réserve **80 %** des jours (800) pour l'entraînement et **20 %** (200) pour le test. Le jeu de test joue le rôle d'un **examen** : le modèle ne le voit jamais pendant l'apprentissage, ce qui permet de vérifier qu'il sait **généraliser** et pas seulement mémoriser.

In [25]:
X = donnees[["Temp_Max", "Temp_Min", "Pluie_Aujourdhui"]]
y = donnees["Pleuvra_Demain"]  # 0 = Beau temps, 1 = Pluie

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 4. Entraînement de l'algorithme Random Forest

Une **forêt aléatoire** est un ensemble de nombreux **arbres de décision** (ici 100). Chaque arbre est entraîné sur un échantillon légèrement différent des données et pose une série de questions du type *« Temp_Max > 32 ? »*, *« Pluie > 0 ? »*.

Pour prédire, **chaque arbre vote** (Pluie ou Beau temps) et la **majorité l'emporte**. Ce vote collectif rend le modèle plus robuste et moins sensible au bruit qu'un arbre unique.

- `n_estimators=100` : nombre d'arbres dans la forêt.
- `random_state=42` : rend l'entraînement reproductible.

In [26]:
print("Entraînement de l'algorithme Random Forest...")
modele = RandomForestClassifier(n_estimators=100, random_state=42)
modele.fit(X_train, y_train)

Entraînement de l'algorithme Random Forest...


RandomForestClassifier(random_state=42)

## 5. Évaluation des performances

On demande au modèle de prédire les 200 jours de test, puis on compare avec la réalité.

In [27]:
predictions = modele.predict(X_test)
exactitude = accuracy_score(y_test, predictions)

print("\n========================================")
print(" Algorithme entraîné avec succès !")
print(f"Exactitude globale (Accuracy) : {exactitude * 100:.2f}%")
print("========================================\n")

print("Rapport détaillé de classification :")
print(classification_report(y_test, predictions))


 Algorithme entraîné avec succès !
Exactitude globale (Accuracy) : 76.50%

Rapport détaillé de classification :
              precision    recall  f1-score   support

           0       0.77      0.88      0.82       124
           1       0.75      0.58      0.65        76

    accuracy                           0.77       200
   macro avg       0.76      0.73      0.74       200
weighted avg       0.76      0.77      0.76       200



### Interprétation des résultats

Résultats obtenus lors de l'exécution de référence :

| Classe | Précision | Rappel | F1-score | Nombre de jours |
|---|---|---|---|---|
| 0 — Beau temps | 0,77 | 0,88 | 0,82 | 124 |
| 1 — Pluie | 0,75 | 0,58 | 0,65 | 76 |
| **Exactitude globale** | | | **76,5 %** | 200 |

**Lecture des indicateurs :**
- **Exactitude (accuracy)** : le modèle a raison sur 76,5 % des jours de test.
- **Précision** (classe Pluie = 0,75) : quand le modèle annonce de la pluie, il a raison 3 fois sur 4.
- **Rappel** (classe Pluie = 0,58) : parmi les jours réellement pluvieux, le modèle en détecte 58 %. Il **manque donc environ 4 jours de pluie sur 10**.
- **F1-score** : moyenne équilibrée entre précision et rappel.

**Pourquoi pas 100 % ?** Les données contiennent un hasard volontaire : même dans les conditions « favorables », il ne pleut que 8 fois sur 10. Le meilleur score théoriquement atteignable est donc d'environ **80 %**. Avec **76,5 %**, le modèle est proche de ce plafond : il a bien retrouvé la règle cachée.

>  Remarque de vocabulaire : en français, *accuracy* se traduit plutôt par **exactitude**, alors que la **précision** (*precision*) est une métrique différente, visible dans le rapport ci-dessus.

## 6. Faire une nouvelle prédiction concrète

On simule les relevés d'une journée : **34 °C** max, **26 °C** min, **5,2 mm** de pluie.

- `predict()` renvoie la classe prédite (0 ou 1).
- `predict_proba()` renvoie la **probabilité** de chaque classe, c'est-à-dire la proportion d'arbres ayant voté pour elle.

Les données sont passées sous forme de `DataFrame` avec les **mêmes noms de colonnes** qu'à l'entraînement, ce qui évite l'avertissement `X does not have valid feature names`.

In [31]:
meteo_aujourdhui = pd.DataFrame(
    [[34.0, 26.0, 5.2]], columns=["Temp_Max", "Temp_Min", "Pluie_Aujourdhui"]
)

prediction = modele.predict(meteo_aujourdhui)
probabilite = modele.predict_proba(meteo_aujourdhui)

print("=== RÉSULTAT DE LA PRÉDICTION ===")
if prediction[0] == 1:
    print(" Résultat : L'algorithme prédit de la PLUIE pour demain !")
else:
    print(" Résultat : L'algorithme prédit du BEAU TEMPS pour demain !")

print(f"Probabilité de pluie : {probabilite[0][1] * 100:.1f}%")

=== RÉSULTAT DE LA PRÉDICTION ===
 Résultat : L'algorithme prédit de la PLUIE pour demain !
Probabilité de pluie : 88.0%


**Analyse :** avec 34 °C et 5,2 mm de pluie, les deux conditions de la règle sont remplies (chaleur > 32 °C **et** pluie > 0). Le modèle prédit donc logiquement de la **pluie**, avec une probabilité d'environ **88 %** lors de l'exécution de référence.

## 7. Sauvegarde et rechargement du modèle

Ré-entraîner le modèle à chaque utilisation serait inutile. Grâce à **Pickle**, on enregistre le modèle entraîné dans le fichier `modele_meteo.pkl`. Il peut ensuite être rechargé instantanément, par exemple dans l'interface graphique, sans refaire l'apprentissage.

Le test de rechargement utilise une journée à 30 °C sans pluie : la condition de chaleur n'étant pas remplie, le modèle prédit **0 (Beau temps)**.

>  Un fichier `.pkl` ne doit être ouvert que s'il provient d'une source de confiance, car il peut exécuter du code au chargement.

In [32]:
import pickle

# 1. Sauvegarder l'algorithme dans un fichier
with open("modele_meteo.pkl", "wb") as fichier:
    pickle.dump(modele, fichier)
print(" L'algorithme a été sauvegardé avec succès dans 'modele_meteo.pkl' !")

# 2. Simulation : le recharger plus tard
with open("modele_meteo.pkl", "rb") as fichier_charge:
    modele_futur = pickle.load(fichier_charge)

test = pd.DataFrame([[30.0, 24.0, 0.0]], columns=["Temp_Max", "Temp_Min", "Pluie_Aujourdhui"])
test_prediction = modele_futur.predict(test)
print(f" Test du modèle chargé : Classe prédite {test_prediction} (0=Beau, 1=Pluie)")

 L'algorithme a été sauvegardé avec succès dans 'modele_meteo.pkl' !
 Test du modèle chargé : Classe prédite [0] (0=Beau, 1=Pluie)


## 8. Application graphique « Météo ML DuvTech » (Tkinter)

Pour rendre le modèle utilisable par tous, on construit une petite **application de bureau** au thème sombre moderne.

**Fonctionnement :**
1. Au lancement, le modèle est chargé depuis `modele_meteo.pkl`.
2. L'utilisateur saisit la température max, la température min et la pluie du jour.
3. **Prédire** (vert) : affiche *PLUIE DEMAIN* (en bleu) ou *BEAU TEMPS* (en jaune) avec le pourcentage de risque de pluie.
4. **Effacer** (rouge) : remet les champs à zéro.
5. Une saisie invalide (lettres, virgule au lieu du point) déclenche un message d'avertissement.

| Couleur | Code | Usage |
|---|---|---|
| Bleu nuit | `#1a202c` | Fond de la fenêtre |
| Gris ardoise | `#2d3748` | Champs de saisie |
| Vert | `#38a169` | Bouton Prédire |
| Rouge | `#e53e3e` | Bouton Effacer |
| Bleu | `#3a86ff` | Résultat « Pluie » |
| Jaune | `#ffbe0b` | Résultat « Beau temps » |

>  La fenêtre s'ouvre à part ; la cellule reste « en cours d'exécution » tant qu'elle n'est pas fermée.

In [ ]:
import pickle
import tkinter as tk
from tkinter import messagebox
import pandas as pd

COLONNES = ["Temp_Max", "Temp_Min", "Pluie_Aujourdhui"]

# 1. Charger le modèle pré-entraîné
try:
    with open("modele_meteo.pkl", "rb") as fichier:
        modele = pickle.load(fichier)
except FileNotFoundError:
    messagebox.showerror(
        "Erreur",
        "Le fichier 'modele_meteo.pkl' est introuvable. Veuillez d'abord réexécuter le code de sauvegarde.",
    )


# 2. Fonction de prédiction
def predire_meteo():
    try:
        tmax = float(entree_tmax.get())
        tmin = float(entree_tmin.get())
        pluie = float(entree_pluie.get())

        donn_utilisateur = pd.DataFrame([[tmax, tmin, pluie]], columns=COLONNES)
        prediction = modele.predict(donn_utilisateur)
        prob_pluie = modele.predict_proba(donn_utilisateur)[0][1] * 100

        if prediction[0] == 1:
            label_resultat.config(
                text=f" PLUIE DEMAIN\n({prob_pluie:.1f}% de risques)", fg="#3a86ff"
            )
        else:
            label_resultat.config(
                text=f" BEAU TEMPS\n({prob_pluie:.1f}% de risques de pluie)", fg="#ffbe0b"
            )
    except ValueError:
        messagebox.showwarning(
            "Attention",
            "Veuillez entrer des nombres valides (utilisez le point '.' pour les décimales).",
        )


# 3. Fonction pour réinitialiser les cases à zéro
def reinitialiser():
    for entree in (entree_tmax, entree_tmin, entree_pluie):
        entree.delete(0, tk.END)
        entree.insert(0, "0.0")
    label_resultat.config(text="Cases réinitialisées. En attente...", fg="#a0aec0")


# 4. Interface graphique personnalisée (thème moderne)
fenetre = tk.Tk()
fenetre.title("Météo ML DuvTech")
fenetre.geometry("380x480")
fenetre.config(bg="#1a202c")

tk.Label(
    fenetre, text="Prédiction Météo Intelligente",
    font=("Helvetica", 14, "bold"), bg="#1a202c", fg="#e2e8f0",
).pack(pady=20)

style_labels = {"bg": "#1a202c", "fg": "#cbd5e0", "font": ("Helvetica", 10)}
style_entries = {
    "justify": "center", "font": ("Helvetica", 11), "bg": "#2d3748", "fg": "white",
    "insertbackground": "white", "bd": 2, "relief": "flat",
}

tk.Label(fenetre, text="Température Maximale (°C)", **style_labels).pack()
entree_tmax = tk.Entry(fenetre, **style_entries)
entree_tmax.insert(0, "34.0")
entree_tmax.pack(pady=8, ipady=4)

tk.Label(fenetre, text="Température Minimale (°C)", **style_labels).pack()
entree_tmin = tk.Entry(fenetre, **style_entries)
entree_tmin.insert(0, "26.0")
entree_tmin.pack(pady=8, ipady=4)

tk.Label(fenetre, text="Pluie enregistrée aujourd'hui (mm)", **style_labels).pack()
entree_pluie = tk.Entry(fenetre, **style_entries)
entree_pluie.insert(0, "5.2")
entree_pluie.pack(pady=8, ipady=4)

cadre_boutons = tk.Frame(fenetre, bg="#1a202c")
cadre_boutons.pack(pady=20)

style_bouton = {"fg": "white", "font": ("Helvetica", 10, "bold"), "relief": "flat",
                "padx": 15, "pady": 5, "cursor": "hand2"}
tk.Button(cadre_boutons, text="Prédire", command=predire_meteo, bg="#38a169",
          **style_bouton).grid(row=0, column=0, padx=10)
tk.Button(cadre_boutons, text="Effacer", command=reinitialiser, bg="#e53e3e",
          **style_bouton).grid(row=0, column=1, padx=10)

label_resultat = tk.Label(
    fenetre, text="En attente de données...",
    font=("Helvetica", 13, "bold"), bg="#1a202c", fg="#a0aec0",
)
label_resultat.pack(pady=15)

fenetre.mainloop()

## 9. Conclusion, limites et perspectives

### Conclusion
Le projet démontre une chaîne complète de Machine Learning : génération de données, entraînement, évaluation, sauvegarde et mise à disposition via une interface graphique. Le modèle Random Forest atteint **76,5 % d'exactitude**, proche du maximum théorique (~80 %) permis par le bruit des données.

### Limites
- **Données simulées** : le modèle apprend une règle inventée, pas la vraie météo haïtienne. Ses prédictions ne doivent pas être utilisées pour des décisions réelles.
- **Peu de variables** : la vraie météo dépend aussi de l'humidité, de la pression, du vent, de la saison et de la région.
- **Rappel faible sur la pluie (58 %)** : le modèle rate encore beaucoup de jours pluvieux, ce qui est gênant si l'objectif est d'alerter.
- **Aucune dimension temporelle** : chaque jour est traité indépendamment, sans tenir compte des jours précédents.

### Perspectives d'amélioration
1. Utiliser de **vraies données** (relevés d'une station locale ou archives météorologiques officielles).
2. Ajouter des variables : **humidité, pression, vent, mois de l'année** (saison des pluies).
3. Améliorer la détection des jours de pluie avec `class_weight="balanced"` ou un **seuil de décision** abaissé.
4. Utiliser la **validation croisée** et comparer d'autres algorithmes (régression logistique, Gradient Boosting).
5. Transformer l'application en **exécutable** (PyInstaller) ou en application web.